In [135]:
#Imports
import pandas as pd
import numpy as np


* Import necessary dataframes

In [136]:
oews_df=pd.read_csv(r'C:\Users\linds\OneDrive\Documents\CII2 Multi-site\oews_2024.csv')
df_sim=pd.read_csv(r'C:\Users\linds\OneDrive\Documents\CII2 Multi-site\merged_similarity_scores_updated.csv')
df_sim2=pd.read_csv(r'C:\Users\linds\OneDrive\Documents\CII2 Multi-site\similarity_scores.csv')
similarity_thresholds=pd.read_csv(r'C:\Users\linds\OneDrive\Documents\CII2 Multi-site\similarity_thresholds.csv')

C:\Users\linds\AppData\Local\Temp\ipykernel_76408\4171048894.py:1: DtypeWarning: Columns (0: jobs_1000, 1: loc_quotient, 2: pct_total, 3: pct_rpt, 4: annual, 5: hourly, 6: naics_code) have mixed types. Specify dtype option on import or set low_memory=False.
  oews_df=pd.read_csv(r'C:\Users\linds\OneDrive\Documents\CII2 Multi-site\oews_2024.csv')


# Cleaning and defining functions
* Note: Based off Shantanu code

In [137]:
#drop duplicates and set index based on occ_code and occ_title--make dict.
def build_occ_code_title_dict(OEWS_df):
    return (
        OEWS_df[["occ_code", "occ_title"]]
        .drop_duplicates()
        .set_index("occ_code")["occ_title"]
        .to_dict()
    )

#Create lookup table of dict of SOC code to occupation titles based on above function
occ_lookup= build_occ_code_title_dict(oews_df)

In [138]:
#Destination wage distributions
def get_destination_wage_row(oews_df, destination_query, area_input):

    wage_cols = ["A_PCT10", "A_PCT25", "A_MEDIAN", "A_PCT75", "A_PCT90"]

    default_row = pd.Series({col: 0 for col in wage_cols}, dtype="float64")

    if oews_df is None or oews_df.empty:
        return default_row

    occ_code = oews_df["OCC_CODE"].astype(str).str.strip()
    area_title = oews_df["AREA_TITLE"].astype(str).str.strip()

    matched = oews_df.loc[
        (occ_code == str(destination_query).strip()) &
        (area_title == str(area_input).strip()),
        wage_cols
    ]

    if matched.empty:
        return default_row

    row = matched.iloc[0].apply(pd.to_numeric, errors="coerce").fillna(0)
    return row.reindex(wage_cols, fill_value=0)

* Merging similarity scores with thresholds

In [139]:
sim_df_merged= df_sim.merge(df_sim2, left_on=["destination_soc","origin_soc"], right_on=["destination","origin"],how="left")
sim_df_merged= sim_df_merged.merge(similarity_thresholds, left_on="destination_soc", right_on="occ_code", how="left")

In [140]:
# Cleaning functions for OEWS data--capitalize and fill na, etc
def Clean_TotalEmployment(a):
  '''
  '''
  if a=='**' or a=='0' or a=='' or a==np.nan:
    return 0
  else:
    return int(a)


def Clean_TotalWages(a):
  if a=='*' or a=='**' or a=='' or a==' ':
      return np.nan
  elif a=='#' or a== '##':
      return 239200
  elif pd.isna(a):
    return float("nan")
  else:
      return int(a)

def clean_oews_subset(df):

  df['TOT_EMP'] = df['tot_emp'].fillna(0)
  df['TOT_EMP']=df['TOT_EMP'].apply(lambda x: Clean_TotalEmployment(x))
  df['A_MEAN']=df['a_mean'].apply(lambda x: Clean_TotalWages(x))


  df["A_PCT10"]= df["a_pct10"].apply(lambda a: Clean_TotalWages(a))
  df["A_PCT25"]= df["a_pct25"].apply(lambda a: Clean_TotalWages(a))
  df["A_MEDIAN"]= df["a_median"].apply(lambda a: Clean_TotalWages(a))
  df["A_PCT75"]= df["a_pct75"].apply(lambda a: Clean_TotalWages(a))
  df["A_PCT90"]= df["a_pct90"].apply(lambda a: Clean_TotalWages(a))

  return df

# Tree Map

In [141]:
#Provide parameters for function calls and filtering
destination_query="51-9111"
cosine_threshold_tree= 0.96
Area_input= "U.S."
median_lb= 0
median_ub= 239200
emp_threshold= 50000

In [142]:
#Create df: all occupations within similarity threshold of destination occupation, with wage and employment data for destination occupation
def tree_map_data_prep(df, destination_query,sim_threshold):
  df_tm= df[(df['destination_soc']==destination_query)& (df['similarity']>=sim_threshold)].reset_index(drop=True).copy()
  return df_tm

In [143]:
#Merge tree map data with OEWS data for occupations within similarity threshold of destination occupation, and calculate transition employment based on similarity score and total employment of origin occupation
def tree_map_data_merge(Area_input, oews_df,df_tm, median_lb, median_ub, emp_threshold):
  OEWS_df= oews_df[oews_df['area_title']==Area_input].copy().reset_index()
  OEWS_df= clean_oews_subset(OEWS_df)
  OEWS_df= OEWS_df[(OEWS_df['A_MEDIAN']>=median_lb) & (OEWS_df['A_MEDIAN']<=median_ub) & (OEWS_df['TOT_EMP']>=emp_threshold) ].copy().reset_index(drop=True)
  final_merged= df_tm.merge(OEWS_df, left_on="origin_soc", right_on="occ_code", how= "inner")

  #Wage comparability times total employment
  final_merged['Transition EMP']= final_merged['Piecwise_wage_comparison_value'] * final_merged['TOT_EMP']
  return final_merged

In [144]:
#Keep only needed data
merged_similarity_scores_updated= sim_df_merged[["destination_soc","origin_soc","similarity","Piecwise_wage_comparison_value", "threshold"]].copy()

In [145]:
#Utilize tree map data prep function
merged_similarity_scores_updated_tree= tree_map_data_prep(merged_similarity_scores_updated,
                                                          destination_query,
                                                          cosine_threshold_tree)


In [146]:
#Utilize tree map data merge function
final_merged_tree= tree_map_data_merge(Area_input, oews_df,merged_similarity_scores_updated_tree, median_lb, median_ub, emp_threshold)

In [147]:
#Head matches Shantanu head
final_merged_tree.head()

,destination_soc,origin_soc,similarity,Piecwise_wage_comparison_value,threshold,index,area,area_title,area_type,prim_state,...,hourly,naics_code,TOT_EMP,A_MEAN,A_PCT10,A_PCT25,A_MEDIAN,A_PCT75,A_PCT90,Transition EMP
0,51-9111,33-2011,0.962303,0.138889,0.94,649,99,U.S.,1,US,...,NaN,0,332240,63890,34490,44180,59530,77410,101330,46144.444444
1,51-9111,33-2011,0.962303,0.138889,0.94,15957,99,U.S.,1,US,...,NaN,99,309750,65370,35660,45760,60460,78480,103110,43020.833333
2,51-9111,33-2011,0.962303,0.138889,0.94,161596,99,U.S.,1,US,...,NaN,999000,309750,65370,35660,45760,60460,78480,103110,43020.833333
3,51-9111,33-2011,0.962303,0.138889,0.94,162813,99,U.S.,1,US,...,NaN,999001,310000,65360,35660,45760,60460,78480,103110,43055.555556
4,51-9111,33-2011,0.962303,0.138889,0.94,167441,99,U.S.,1,US,...,NaN,999300,292050,65530,35560,45650,60360,78890,103960,40562.500000


In [148]:
import plotly.graph_objects as go

# -----------------------------
# SOC labels
# -----------------------------
soc_labels = {
    "11": "Management",
    "13": "Business & Financial Operations",
    "15": "Computer & Mathematical",
    "17": "Architecture & Engineering",
    "19": "Life / Physical / Social Science",
    "21": "Community & Social Service",
    "23": "Legal",
    "25": "Education / Library",
    "27": "Arts / Design / Media / Sports",
    "29": "Healthcare Practitioners & Technical",
    "31": "Healthcare Support",
    "33": "Protective Service",
    "35": "Food Preparation & Serving",
    "37": "Building / Grounds Cleaning",
    "39": "Personal Care & Service",
    "41": "Sales",
    "43": "Office & Administrative Support",
    "45": "Farming / Fishing / Forestry",
    "47": "Construction & Extraction",
    "49": "Installation / Maintenance / Repair",
    "51": "Production",
    "53": "Transportation / Material Moving"
}

# -----------------------------
# Predefined color dictionary
# -----------------------------
soc_color_map = {
    "11": "#4E79A7",
    "13": "#F28E2B",
    "15": "#E15759",
    "17": "#76B7B2",
    "19": "#59A14F",
    "21": "#EDC948",
    "23": "#B07AA1",
    "25": "#FF9DA7",
    "27": "#9C755F",
    "29": "#BAB0AC",
    "31": "#86BCB6",
    "33": "#D37295",
    "35": "#FABFD2",
    "37": "#8CD17D",
    "39": "#B6992D",
    "41": "#499894",
    "43": "#E15759",
    "45": "#79706E",
    "47": "#D4A6C8",
    "49": "#A0CBE8",
    "51": "#FFBE7D",
    "53": "#8CD3FF"
}

# -----------------------------
# Helper: wrap text
# -----------------------------
def wrap_every_n_words(text, n=3):
    words = str(text).split()
    return "<br>".join(
        [" ".join(words[i:i+n]) for i in range(0, len(words), n)]
    )




# -----------------------------
# Main plotting function
# -----------------------------
def plot_occ_treemap(
    df,
    destination_query,
    occ_lookup,
    value_col="Transition EMP",
    code_col="occ_code",
    title_col="occ_title",
    area_col="area_title",
    aggregate_duplicates=True,
    height=800,
    width=1400,
    title=" ",#"Treemap of OCC_CODE / OCC_TITLE by Piecewise Wage Comparison Value",
    color_map=None,
    unknown_color="#D3D3D3"
):
    """
    Build a Plotly treemap where each box is an OCC_CODE + OCC_TITLE pair,
    sized by `value_col`, and colored by the first two digits of `code_col`.

    A right-side information panel (20% width of total figure) is added
    to display:
      - Occupation of interest
      - OCC_TITLE corresponding to destination_query
      - Area title
      - Major group color scheme

    Parameters
    ----------
    df : pandas.DataFrame
        Input dataframe used for plotting.
    destination_query : str
        OCC_CODE whose occupation title should be shown in the right panel.
    occ_lookup : dict
        Dictionary mapping OCC_CODE -> OCC_TITLE.
        Build this using build_occ_code_title_dict(OEWS_df).
    value_col : str
        Column containing numeric values used for box size.
    code_col : str
        OCC code column.
    title_col : str
        OCC title column.
    area_col : str
        Area title column. Expected to contain only one unique value in df.
    aggregate_duplicates : bool
        If True, duplicate OCC_CODE/OCC_TITLE pairs are summed.
    height : int
        Figure height.
    width : int
        Figure width.
    title : str
        Plot title.
    color_map : dict or None
        Mapping from first two digits of OCC_CODE to color.
    unknown_color : str
        Fallback color if first two digits are not found.

    Returns
    -------
    plotly.graph_objects.Figure
        Plotly treemap figure.
    """

    if color_map is None:
        color_map = soc_color_map

    required_cols = [code_col, title_col, value_col, area_col]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns in df: {missing}")

    # -----------------------------
    # Clean and prepare plotting df
    # -----------------------------
    plot_df = df[[code_col, title_col, value_col, area_col]].copy()
    plot_df[code_col] = plot_df[code_col].astype(str).str.strip()
    plot_df[title_col] = plot_df[title_col].astype(str).str.strip()
    plot_df[area_col] = plot_df[area_col].astype(str).str.strip()
    plot_df[value_col] = pd.to_numeric(plot_df[value_col], errors="coerce")

    plot_df = plot_df.dropna(subset=[value_col, code_col, title_col])

    if aggregate_duplicates:
        plot_df = (
            plot_df.groupby([code_col, title_col, area_col], as_index=False)[value_col]
            .sum()
        )

    plot_df = plot_df.sort_values(by=value_col, ascending=False).reset_index(drop=True)

    # -----------------------------
    # Extract first two digits for coloring
    # -----------------------------
    plot_df["soc_2digit"] = (
        plot_df[code_col]
        .astype(str)
        .str.extract(r"(\d{2})", expand=False)
    )

    plot_df["soc_label"] = plot_df["soc_2digit"].map(soc_labels).fillna("Unknown")
    plot_df["color"] = plot_df["soc_2digit"].map(color_map).fillna(unknown_color)

    # -----------------------------
    # Labels inside treemap
    # -----------------------------
    plot_df["label"] = (
        plot_df[code_col].astype(str)
        + "<br>"
        + plot_df[title_col].astype(str).apply(lambda x: wrap_every_n_words(x, n=3))
    )

    # -----------------------------
    # Destination occupation title
    # -----------------------------
    destination_query = str(destination_query).strip()
    destination_title = occ_lookup.get(destination_query, "Code not found in OEWS lookup")

    # -----------------------------
    # Area title (expect exactly one unique value)
    # -----------------------------
    area_values = [x for x in plot_df[area_col].dropna().unique() if str(x).strip() != ""]

    if len(area_values) == 0:
        area_title_value = "Unknown"
    elif len(area_values) == 1:
        area_title_value = area_values[0]
    else:
        raise ValueError(
            f"Expected one unique {area_col} in df, found: {list(area_values)}"
        )

    # -----------------------------
    # Build compact legend text
    # -----------------------------
    legend_lines = []
    for soc_code, soc_name in soc_labels.items():
        legend_color = color_map.get(soc_code, unknown_color)
        legend_lines.append(
            f'<span style="color:{legend_color};"><b>■</b></span> '
            f'{soc_code}: {soc_name}'
        )
    legend_text = "<br>".join(legend_lines)

    # -----------------------------
    # Make treemap
    # Left 80% = treemap
    # Right 20% = information panel
    # -----------------------------
    fig = go.Figure(
        go.Treemap(
            labels=plot_df["label"],
            parents=[""] * len(plot_df),
            values=plot_df[value_col],
            marker=dict(colors=plot_df["color"]),
            customdata=plot_df[[code_col, title_col, value_col, "soc_2digit", "soc_label"]],
            sort=False,
            textinfo="label",
            domain=dict(x=[0.0, 0.79], y=[0.0, 1.0]),
            hovertemplate=(
                "<b>%{customdata[0]}</b><br>"
                "%{customdata[1]}<br>"
                "SOC Group: %{customdata[3]} - %{customdata[4]}<br>"
                f"{value_col}: " + "%{customdata[2]:,.4f}"
                "<extra></extra>"
            ),
        )
    )

    # -----------------------------
    # Right side panel background
    # -----------------------------
    fig.add_shape(
        type="rect",
        xref="paper",
        yref="paper",
        x0=0.79,
        x1=1.00,
        y0=0.00,
        y1=0.97,
        line=dict(color="lightgray", width=1),
        fillcolor="rgba(245,245,245,0.95)"
    )

    # separator line
    fig.add_shape(
        type="line",
        xref="paper",
        yref="paper",
        x0=0.79,
        x1=0.79,
        y0=0.03,
        y1=0.97,
        line=dict(color="gray", width=1)
    )

    # -----------------------------
    # Right-side annotations
    # -----------------------------
    fig.add_annotation(
        x=0.81, y=0.96,
        xref="paper", yref="paper",
        xanchor="left", yanchor="top",
        align="left",
        showarrow=False,
        text="<b>Occupation of interest</b>",
        font=dict(size=16, color="black")
    )

    fig.add_annotation(
        x=0.81, y=0.92,
        xref="paper", yref="paper",
        xanchor="left", yanchor="top",
        align="left",
        showarrow=False,
        text=wrap_every_n_words(destination_title, n=3),
        font=dict(size=13, color="black")
    )

    fig.add_annotation(
        x=0.81, y=0.76,
        xref="paper", yref="paper",
        xanchor="left", yanchor="top",
        align="left",
        showarrow=False,
        text="<b>Area title</b>",
        font=dict(size=16, color="black")
    )

    fig.add_annotation(
        x=0.81, y=0.72,
        xref="paper", yref="paper",
        xanchor="left", yanchor="top",
        align="left",
        showarrow=False,
        text=wrap_every_n_words(area_title_value, n=4),
        font=dict(size=13, color="black")
    )

    fig.add_annotation(
        x=0.81, y=0.58,
        xref="paper", yref="paper",
        xanchor="left", yanchor="top",
        align="left",
        showarrow=False,
        text="<b>Major groups</b>",
        font=dict(size=16, color="black")
    )

    fig.add_annotation(
        x=0.81, y=0.54,
        xref="paper", yref="paper",
        xanchor="left", yanchor="top",
        align="left",
        text=legend_text,
        font=dict(size=11, color="black"),
        showarrow=False
    )

    fig.update_layout(
        title=title,
        height=height,
        width=width,
        margin=dict(t=50, l=10, r=10, b=10)
    )

    return fig

In [149]:
fig = plot_occ_treemap(final_merged_tree,destination_query,occ_lookup)
fig.show()

# Measure if Occupations are Pulling from the Same Labor Pool

In [150]:
def get_origin_pool(destination_query, Area_input, cosine_threshold_tree,
                    oews_df, merged_similarity_scores_updated,
                    median_lb, median_ub, emp_threshold):
    df_tm = tree_map_data_prep(
        merged_similarity_scores_updated,
        destination_query,
        cosine_threshold_tree
    )

    final_merged_tree = tree_map_data_merge(
        Area_input,
        oews_df,
        df_tm,
        median_lb,
        median_ub,
        emp_threshold
    )

    pool = (
        final_merged_tree.groupby("origin_soc", as_index=True)["Transition EMP"]
        .sum()
    )

    return pool


In [151]:
def weighted_jaccard_from_pools(pool_a, pool_b):
    all_origins = pool_a.index.union(pool_b.index)
    a = pool_a.reindex(all_origins, fill_value=0)
    b = pool_b.reindex(all_origins, fill_value=0)

    return (np.minimum(a, b).sum() / np.maximum(a, b).sum())


In [152]:
#Sample dest_a and dest_b
dest_a = "51-9111"
dest_b = "51-9199"


In [153]:
pool_a = get_origin_pool(dest_a, Area_input, cosine_threshold_tree,
                         oews_df, merged_similarity_scores_updated,
                         median_lb, median_ub, emp_threshold)

pool_b = get_origin_pool(dest_b, Area_input, cosine_threshold_tree,
                         oews_df, merged_similarity_scores_updated,
                         median_lb, median_ub, emp_threshold)

score = weighted_jaccard_from_pools(pool_a, pool_b)
print(score)


0.31019210701746547


# Visualizing Jaccard Similarity in X occupations

In [154]:
def build_origin_transition_pool(destination_query, Area_input, oews_df, merged_similarity_scores_updated,
                                 cosine_threshold_tree, median_lb, median_ub, emp_threshold):
    """Build the origin occupation pool for one destination occupation."""
    df_tm = tree_map_data_prep(
        merged_similarity_scores_updated,
        destination_query,
        cosine_threshold_tree
    )

    final_merged_tree = tree_map_data_merge(
        Area_input,
        oews_df,
        df_tm,
        median_lb,
        median_ub,
        emp_threshold
    )

    pool = (
        final_merged_tree.groupby("origin_soc")["Transition EMP"]
        .sum()
        .sort_values(ascending=False)
    )

    return pool


def compare_destination_overlap(dest_a, dest_b, Area_input, oews_df, merged_similarity_scores_updated,
                                cosine_threshold_tree, median_lb, median_ub, emp_threshold):
    """Compute weighted Jaccard overlap between two destination occupations."""
    pool_a = build_origin_transition_pool(
        dest_a, Area_input, oews_df, merged_similarity_scores_updated,
        cosine_threshold_tree, median_lb, median_ub, emp_threshold
    )
    pool_b = build_origin_transition_pool(
        dest_b, Area_input, oews_df, merged_similarity_scores_updated,
        cosine_threshold_tree, median_lb, median_ub, emp_threshold
    )

    all_origins = pool_a.index.union(pool_b.index)

    aligned = pd.DataFrame({
        "pool_a": pool_a.reindex(all_origins, fill_value=0),
        "pool_b": pool_b.reindex(all_origins, fill_value=0)
    })

    numerator = aligned[["pool_a", "pool_b"]].min(axis=1).sum()
    denominator = aligned[["pool_a", "pool_b"]].max(axis=1).sum()

    weighted_jaccard = numerator / denominator if denominator else np.nan
    return weighted_jaccard


def build_weighted_jaccard_table(destination_queries, Area_input, oews_df, merged_similarity_scores_updated,
                                 cosine_threshold_tree, median_lb, median_ub, emp_threshold, occ_lookup=None):
    """Return a long table of pairwise weighted Jaccard similarities."""
    records = []

    for i, dest_a in enumerate(destination_queries):
        for dest_b in destination_queries[i+1:]:
            wj = compare_destination_overlap(
                dest_a,
                dest_b,
                Area_input,
                oews_df,
                merged_similarity_scores_updated,
                cosine_threshold_tree,
                median_lb,
                median_ub,
                emp_threshold
            )

            records.append({
                "destination_a": dest_a,
                "destination_a_title": occ_lookup.get(dest_a, "") if occ_lookup else "",
                "destination_b": dest_b,
                "destination_b_title": occ_lookup.get(dest_b, "") if occ_lookup else "",
                "weighted_jaccard": wj
            })

    return (
        pd.DataFrame(records)
        .sort_values("weighted_jaccard", ascending=False)
        .reset_index(drop=True)
    )


In [155]:
def build_overlap_matrix(destination_queries, Area_input, oews_df, merged_similarity_scores_updated,
                         cosine_threshold_tree, median_lb, median_ub, emp_threshold):
    """Build a pairwise weighted Jaccard matrix for destination occupations."""
    destination_queries = list(dict.fromkeys(destination_queries))
    overlap_matrix = pd.DataFrame(index=destination_queries, columns=destination_queries, dtype=float)

    for dest_a in destination_queries:
        for dest_b in destination_queries:
            if pd.isna(overlap_matrix.loc[dest_a, dest_b]):
                weighted_jaccard = compare_destination_overlap(
                    dest_a,
                    dest_b,
                    Area_input,
                    oews_df,
                    merged_similarity_scores_updated,
                    cosine_threshold_tree,
                    median_lb,
                    median_ub,
                    emp_threshold
                )

                overlap_matrix.loc[dest_a, dest_b] = weighted_jaccard
                overlap_matrix.loc[dest_b, dest_a] = weighted_jaccard

    return overlap_matrix


def plot_overlap_heatmap(overlap_matrix, occ_lookup=None,
                         title="Weighted Jaccard overlap across destination occupations"):
    """Plot a heat map of pairwise weighted Jaccard values."""
    if occ_lookup is None:
        labels = list(overlap_matrix.index)
    else:
        labels = [f"{code}<br>{occ_lookup.get(code, 'Unknown occupation')}" for code in overlap_matrix.index]

    fig = go.Figure(
        data=go.Heatmap(
            z=overlap_matrix.values,
            x=labels,
            y=labels,
            zmin=0,
            zmax=1,
            colorscale="Blues",
            colorbar=dict(title="Weighted<br>Jaccard"),
            hovertemplate=(
                "Destination A: %{y}<br>"
                "Destination B: %{x}<br>"
                "Weighted Jaccard: %{z:.3f}<extra></extra>"
            )
        )
    )

    fig.update_layout(
        title=title,
        height=850,
        width=950,
        xaxis=dict(side="bottom"),
        margin=dict(t=80, l=20, r=20, b=20)
    )

    return fig


In [156]:
Area_input = "Chambersburg, PA"
emp_threshold = 100

destination_queries_compare = [
    "11-3051", 
    "13-1081", 
    "17-2071", 
    "17-2112", 
]

overlap_matrix_chambersburg = build_overlap_matrix(
    destination_queries_compare,
    Area_input,
    oews_df,
    merged_similarity_scores_updated,
    cosine_threshold_tree,
    median_lb,
    median_ub,
    emp_threshold
)

display(overlap_matrix_chambersburg)

fig_overlap_chambersburg = plot_overlap_heatmap(
    overlap_matrix_chambersburg,
    occ_lookup=occ_lookup,
    title="Weighted Jaccard overlap in Chambersburg, PA"
)
fig_overlap_chambersburg.show()


,11-3051,13-1081,17-2071,17-2112
11-3051,1.000000,0.114777,0.038972,0.226599
13-1081,0.114777,1.000000,0.034657,0.239094
17-2071,0.038972,0.034657,1.000000,0.283196
17-2112,0.226599,0.239094,0.283196,1.000000


In [157]:
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform

def reorder_overlap_matrix(overlap_matrix):
    """Reorder a similarity matrix so the most similar occupations are adjacent."""
    similarity = overlap_matrix.fillna(0).copy()

    # Convert similarity to distance
    distance = 1 - similarity

    # Force a clean diagonal for clustering
    for i in range(len(distance)):
        distance.iloc[i, i] = 0

    condensed = squareform(distance.values, checks=False)
    linkage_matrix = linkage(condensed, method="average")
    order = leaves_list(linkage_matrix)

    ordered_codes = similarity.index[order]
    ordered_matrix = similarity.loc[ordered_codes, ordered_codes]

    return ordered_matrix


In [158]:
ordered_overlap_matrix_chambersburg = reorder_overlap_matrix(overlap_matrix_chambersburg)

fig_overlap_chambersburg = plot_overlap_heatmap(
    ordered_overlap_matrix_chambersburg,
    occ_lookup=occ_lookup,
    title="Weighted Jaccard overlap in Chambersburg, PA"
)
fig_overlap_chambersburg.show()
